In [ ]:
import json
import re
import fitz  # PyMuPDF for PDF parsing
import requests
from typing import Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

from utils.mistral_client import call_llm, safe_json


# Shared state

In [7]:
class ResumeState(TypedDict):
    # ── Inputs ───────────────────────────────────────────
    raw_resume:         str        # raw text, from PDF or direct input
    raw_jd:             str        # raw job description text

    # ── Parsed Resume ────────────────────────────────────
    resume_sections:    Optional[dict]
    # Structure:
    # {
    #   "name": str,
    #   "title": str,
    #   "contacts": {"phone": str, "email": str, "location": str},
    #   "socials":  {"linkedin": str, "github": str, "other": []},
    #   "summary":  str,
    #   "skills":   [str],
    #   "experience": [{"role": str, "company": str, "period": str, "bullets": [str]}],
    #   "projects":  [{"name": str, "description": str}],
    #   "education": [{"degree": str, "institution": str, "year": str, "notes": str}],
    #   "certifications": [str],
    #   "publications": [str]
    # }

    # ── Parsed JD ────────────────────────────────────────
    jd_analysis:        Optional[dict]
    # Structure:
    # {
    #   "job_title":    str,
    #   "summary":      str,
    #   "required_skills":   [str],
    #   "preferred_skills":  [str],
    #   "experience_needed": str,
    #   "responsibilities":  [str],
    #   "personality":       [str]   # e.g. "self-starter", "collaborative"
    # }

    # ── Gap Analysis ─────────────────────────────────────
    gap_report:         Optional[dict]
    # Structure:
    # {
    #   "matching_skills":  [str],
    #   "missing_skills":   [str],
    #   "weak_sections":    [str],
    #   "reorder_priority": [str],   # section names in priority order
    #   "suggestions":      [str]    # specific rewrite suggestions
    # }

    # ── Refined Resume ───────────────────────────────────
    refined_resume:     Optional[dict]  # same structure as resume_sections

    # ── Output ───────────────────────────────────────────
    output_pdf_path:    Optional[str]
    error:              Optional[str]

In [ ]:
def load_resume(source: str) -> str:
    """
    Load a resume from a file path or return the raw text directly.

    If the source path ends with .pdf, the PDF is parsed using PyMuPDF.
    Otherwise the source is returned as plain text.
    """
    if source.strip().endswith(".pdf"):
        try:
            doc = fitz.open(source)
            return "\n".join(page.get_text() for page in doc)
        except Exception as e:
            return f"PDF read failed: {e}"
    else:
        return source  # already plain text

# Example test paths:
# resume_text = load_resume("Arjun_V_Resume.pdf")
# resume_text = load_resume("paste your resume text here...")
# print(resume_text[:500])


In [ ]:
RESUME_PARSER_SYSTEM = """You are a resume parsing agent.
Extract all sections from the resume text provided.
Return ONLY a valid JSON object with this exact structure — no markdown, no explanation:

{
  "name": "",
  "title": "",
  "contacts": {"phone": "", "email": "", "location": ""},
  "socials": {"linkedin": "", "github": "", "other": []},
  "summary": "",
  "skills": [],
  "experience": [
    {"role": "", "company": "", "period": "", "bullets": []}
  ],
  "projects": [
    {"name": "", "description": ""}
  ],
  "education": [
    {"degree": "", "institution": "", "year": "", "notes": ""}
  ],
  "certifications": [],
  "publications": []
}

Rules:
- skills must be a flat list of individual skill strings
- experience bullets must be individual strings, one per bullet
- If a section is absent from the resume, use empty string or empty list
- Extract exactly what is written — do not infer or add anything"""

def resume_parser_node(state: ResumeState) -> ResumeState:
    """Parse the raw resume text into structured JSON fields."""
    print("[Resume Parser] Extracting sections...")

    raw = call_llm(RESUME_PARSER_SYSTEM, state["raw_resume"])
    parsed = safe_json(raw)

    if not parsed:
        return {**state, "error": "Resume parser failed to return valid JSON"}

    print(f"[Resume Parser] Extracted: {list(parsed.keys())}")
    return {**state, "resume_sections": parsed}


In [ ]:
JD_PARSER_SYSTEM = """You are a job description analysis agent.
Analyse the job description and extract structured requirements.
Return ONLY a valid JSON object — no markdown, no explanation:

{
  "job_title": "",
  "summary": "",
  "required_skills": [],
  "preferred_skills": [],
  "experience_needed": "",
  "responsibilities": [],
  "personality": []
}

Rules:
- required_skills: must-have technical skills explicitly stated
- preferred_skills: nice-to-have or implied skills
- experience_needed: years and domain as a string e.g. "5+ years ML engineering"
- personality: traits implied or stated e.g. "self-starter", "collaborative", "detail-oriented"
- responsibilities: key job duties as short strings"""

def jd_parser_node(state: ResumeState) -> ResumeState:
    """Parse the raw job description into a structured requirements object."""
    print("[JD Parser] Analysing job description...")

    raw = call_llm(JD_PARSER_SYSTEM, state["raw_jd"])
    parsed = safe_json(raw)

    if not parsed:
        return {**state, "error": "JD parser failed to return valid JSON"}

    print(f"[JD Parser] Role: {parsed.get('job_title')}")
    print(f"[JD Parser] Required skills: {parsed.get('required_skills', [])[:5]}")
    return {**state, "jd_analysis": parsed}


In [ ]:
GAP_ANALYSER_SYSTEM = """You are a resume gap analysis agent.
You will receive a parsed resume and a parsed job description.
Your job is to do a differential analysis and identify exactly what needs to change.
Return ONLY a valid JSON object — no markdown, no explanation:

{
  "matching_skills": [],
  "missing_skills": [],
  "weak_sections": [],
  "reorder_priority": [],
  "suggestions": []
}

Rules:
- matching_skills: skills present in BOTH resume and JD
- missing_skills: skills in JD that are completely absent from resume
- weak_sections: section names where resume is thin relative to JD requirements
- reorder_priority: list section names in order of importance FOR THIS JD
  e.g. ["experience", "skills", "projects", "publications", "education", "certifications"]
- suggestions: specific actionable rewrite instructions, one per string
  e.g. "Rewrite Publicis Sapient summary bullet to mention agentic AI explicitly"
  e.g. "Add LangGraph to skills — candidate has used it in projects"
  e.g. "Move projects section above education — JD values hands-on work""" 

def gap_analyser_node(state: ResumeState) -> ResumeState:
    """Compare resume content against JD requirements and identify gaps."""
    print("[Gap Analyser] Running differential analysis...")

    if state.get("error"):
        return state

    payload = json.dumps({
        "resume":  state["resume_sections"],
        "jd":      state["jd_analysis"]
    }, indent=2)

    raw = call_llm(GAP_ANALYSER_SYSTEM, payload)
    parsed = safe_json(raw)

    if not parsed:
        return {**state, "error": "Gap analyser failed"}

    print(f"[Gap Analyser] Matching skills : {parsed.get('matching_skills', [])}")
    print(f"[Gap Analyser] Missing skills  : {parsed.get('missing_skills', [])}")
    print(f"[Gap Analyser] Weak sections   : {parsed.get('weak_sections', [])}")
    return {**state, "gap_report": parsed}


In [ ]:
REFINEMENT_SYSTEM = """You are a senior resume refinement agent.
You will receive the original parsed resume, the JD analysis, and a gap report with suggestions.
Your job is to produce a refined resume that:
1. Rewrites experience bullets to naturally reflect JD keywords where honest
2. Reorders sections according to reorder_priority from the gap report
3. Strengthens weak sections based on suggestions
4. Adds missing skills to the skills list ONLY if they are genuinely evidenced in experience
5. Sharpens the summary to align with the job title and requirements

Return ONLY the refined resume as a valid JSON object in the SAME structure as the input resume.
No markdown. No explanation. Only JSON."""

def refinement_agent_node(state: ResumeState) -> ResumeState:
    """Refine the parsed resume using the job description and gap analysis.

    This step rewrites the resume content rather than extracting it.
    """
    print("[Refinement Agent] Rewriting resume...")

    if state.get("error"):
        return state

    payload = json.dumps({
        "original_resume": state["resume_sections"],
        "jd_analysis":     state["jd_analysis"],
        "gap_report":      state["gap_report"]
    }, indent=2)

    # Higher temperature — we want rewriting, not just extraction
    raw = call_llm(REFINEMENT_SYSTEM, payload, temperature=0.4)
    refined = safe_json(raw)

    if not refined:
        print("[Refinement Agent] JSON parse failed — using original resume")
        refined = state["resume_sections"]

    print("[Refinement Agent] Refinement complete")
    return {**state, "refined_resume": refined}


In [ ]:
NAVY   = (0.106, 0.227, 0.361)
ACCENT = (0.145, 0.388, 0.922)
GRAY   = (0.294, 0.333, 0.408)
BLACK  = (0.122, 0.122, 0.122)


def pdf_builder_node(state: ResumeState) -> ResumeState:
    """Build a printable PDF resume from the refined resume JSON."""
    print("[PDF Builder] Building PDF...")

    if state.get("error"):
        return state

    r  = state["refined_resume"]
    gp = state.get("gap_report", {})

    # Use the recommended section order from the gap report if present.
    section_order = gp.get("reorder_priority", 
        ["experience", "skills", "projects", "publications", "certifications", "education"])

    doc  = fitz.open()
    page = doc.new_page(width=595, height=842)  # A4 page size

    margin_l = 45
    margin_r = 550
    y        = 50
    line_h   = 13

    def new_page_if_needed(y, needed=40):
        """Start a new page if there is not enough vertical space left."""
        nonlocal page
        if y + needed > 810:
            page = doc.new_page(width=595, height=842)
            return 50
        return y

    def draw_text(text, x, y, size=9, color=BLACK, bold=False):
        fontname = "helv" if not bold else "hebo"
        page.insert_text((x, y), text, fontsize=size, color=color, fontname=fontname)

    def draw_line(y, color=ACCENT, width=0.5):
        page.draw_line((margin_l, y), (margin_r, y), color=color, width=width)

    def section_header(title, y):
        """Render a section header and separator line."""
        y = new_page_if_needed(y, 30)
        y += 8
        draw_text(title.upper(), margin_l, y, size=9, color=NAVY, bold=True)
        y += 3
        draw_line(y, color=ACCENT)
        return y + 10

    # ── NAME + TITLE + CONTACTS ─────────────────────────
    name = r.get("name", "")
    page.insert_text((margin_l, y), name, fontsize=22, color=NAVY, fontname="hebo")
    y += 20

    title = r.get("title", "")
    page.insert_text((margin_l, y), title, fontsize=10, color=ACCENT, fontname="helv")
    y += 14

    c = r.get("contacts", {})
    s = r.get("socials", {})
    contact_parts = [
        c.get("email", ""), c.get("phone", ""), c.get("location", ""),
        s.get("linkedin", ""), s.get("github", "")
    ]
    contact_line = "  ·  ".join(p for p in contact_parts if p)
    draw_text(contact_line, margin_l, y, size=8, color=GRAY)
    y += 18
    draw_line(y, color=(0.8, 0.8, 0.8), width=0.3)
    y += 10

    # ── SUMMARY SECTION ─────────────────────────────────
    if r.get("summary"):
        y = section_header("Professional Summary", y)
        words  = r["summary"].split()
        line   = ""
        for word in words:
            test = line + " " + word if line else word
            if len(test) > 95:
                draw_text(line, margin_l, y, size=8.5)
                y += line_h
                y  = new_page_if_needed(y)
                line = word
            else:
                line = test
        if line:
            draw_text(line, margin_l, y, size=8.5)
            y += line_h + 4

    # ── SKILLS SECTION ──────────────────────────────────
    if r.get("skills"):
        y = section_header("Technical Skills", y)
        skills_text = "  ·  ".join(r["skills"])
        words = skills_text.split()
        line  = ""
        for word in words:
            test = line + " " + word if line else word
            if len(test) > 95:
                draw_text(line, margin_l, y, size=8.5)
                y += line_h
                y  = new_page_if_needed(y)
                line = word
            else:
                line = test
        if line:
            draw_text(line, margin_l, y, size=8.5)
            y += line_h + 4

    # ── DYNAMIC SECTIONS IN PRIORITY ORDER ──────────────
    for section in section_order:

        if section == "experience" and r.get("experience"):
            y = section_header("Professional Experience", y)
            for exp in r["experience"]:
                y = new_page_if_needed(y, 30)
                role_line = f"{exp.get('role','')}  ·  {exp.get('company','')}  |  {exp.get('period','')}"
                draw_text(role_line, margin_l, y, size=9, bold=True, color=NAVY)
                y += line_h
                for bullet in exp.get("bullets", []):
                    y = new_page_if_needed(y, 14)
                    draw_text("▸", margin_l, y, size=8, color=ACCENT)
                    words = bullet.split()
                    line  = ""
                    bx    = margin_l + 12
                    for word in words:
                        test = line + " " + word if line else word
                        if len(test) > 88:
                            draw_text(line, bx, y, size=8.5)
                            y   += line_h
                            y    = new_page_if_needed(y)
                            line = word
                        else:
                            line = test
                    if line:
                        draw_text(line, bx, y, size=8.5)
                        y += line_h
                y += 5

        elif section == "projects" and r.get("projects"):
            y = section_header("Selected Projects", y)
            for proj in r["projects"]:
                y = new_page_if_needed(y, 25)
                draw_text(proj.get("name", ""), margin_l, y, size=9, bold=True, color=NAVY)
                y += line_h
                desc  = proj.get("description", "")
                words = desc.split()
                line  = ""
                for word in words:
                    test = line + " " + word if line else word
                    if len(test) > 92:
                        draw_text(line, margin_l + 8, y, size=8.5)
                        y   += line_h
                        y    = new_page_if_needed(y)
                        line = word
                    else:
                        line = test
                if line:
                    draw_text(line, margin_l + 8, y, size=8.5)
                y += line_h + 4

        elif section == "publications" and r.get("publications"):
            y = section_header("Research & Technical Writing", y)
            for pub in r["publications"]:
                y = new_page_if_needed(y, 14)
                draw_text("▸", margin_l, y, size=8, color=ACCENT)
                words = pub.split()
                line  = ""
                bx    = margin_l + 12
                for word in words:
                    test = line + " " + word if line else word
                    if len(test) > 88:
                        draw_text(line, bx, y, size=8.5)
                        y   += line_h
                        y    = new_page_if_needed(y)
                        line = word
                    else:
                        line = test
                if line:
                    draw_text(line, bx, y, size=8.5)
                y += line_h + 2

        elif section == "education" and r.get("education"):
            y = section_header("Education", y)
            for edu in r["education"]:
                y = new_page_if_needed(y, 25)
                draw_text(
                    f"{edu.get('degree','')}  —  {edu.get('institution','')}  |  {edu.get('year','')}",
                    margin_l, y, size=9, bold=True, color=NAVY
                )
                y += line_h
                if edu.get("notes"):
                    draw_text(edu["notes"], margin_l + 8, y, size=8.5, color=GRAY)
                    y += line_h
                y += 4

        elif section == "certifications" and r.get("certifications"):
            y = section_header("Certifications", y)
            for cert in r["certifications"]:
                y = new_page_if_needed(y, 14)
                draw_text(f"▸  {cert}", margin_l, y, size=8.5)
                y += line_h

    output_path = "refined_resume.pdf"
    doc.save(output_path)
    print(f"[PDF Builder] Saved to {output_path}")
    return {**state, "output_pdf_path": output_path}


In [ ]:
def check_for_errors(state: ResumeState) -> str:
    """Return the next graph route depending on whether an error exists."""
    if state.get("error"):
        return "error_node"
    return "continue"


def error_node(state: ResumeState) -> ResumeState:
    """Terminal node for pipeline failure reporting."""
    print(f"[ERROR] Pipeline stopped: {state.get('error')}")
    return state


In [ ]:
builder = StateGraph(ResumeState)

# Register each processing node in the graph pipeline.
builder.add_node("resume_parser",    resume_parser_node)
builder.add_node("jd_parser",        jd_parser_node)
builder.add_node("gap_analyser",     gap_analyser_node)
builder.add_node("refinement_agent", refinement_agent_node)
builder.add_node("pdf_builder",      pdf_builder_node)
builder.add_node("error_node",       error_node)

# Entry point for the resume customization workflow.
builder.set_entry_point("resume_parser")

# Conditional edges: resume_parser and jd_parser may route to error_node.
builder.add_conditional_edges("resume_parser", check_for_errors,
    {"continue": "jd_parser", "error_node": "error_node"})

builder.add_conditional_edges("jd_parser", check_for_errors,
    {"continue": "gap_analyser", "error_node": "error_node"})

# Continue through the remaining stages once parsing succeeds.
builder.add_edge("gap_analyser",     "refinement_agent")
builder.add_edge("refinement_agent", "pdf_builder")
builder.add_edge("pdf_builder",      END)
builder.add_edge("error_node",       END)

app = builder.compile()
print("Graph compiled")


Graph compiled


In [ ]:
# Load a sample resume and job description to run the full pipeline.
# Replace these with your own resume / job description content.

# Option A: load from a local PDF file
# resume_text = load_resume("path/to/your_resume.pdf")

# Option B: paste the resume text directly
resume_text = load_resume("/Users/arjunv/Desktop/projects/MistralDesk/data/Arjun_V-DS-ML-V1 (1).pdf")

jd_text = """
About the job
Note: Please apply only if you can provide the applicable details such as Candidate PF Number, ESI/Medical Insurance, and EPFO Registration.



Role: GenAI & Agentic AI
Experience: Mind level 5–7 years
Role- Sr GenAI & Agentic AI
Experience: 9+ yrs 

Location- Onsite Bangalore
About the Role

Generative AI (GenAI), Agentic AI, and modern LLM (Large Language Model) ecosystems. The ideal candidate will have hands-on experience with LangChain, LangGraph, MCP, AgentOps, RAG pipelines, fine-tuning models, and MLOps practices, along with proficiency in cloud deployment (AWS, Azure AI, Bedrock, etc.). You will be responsible for building, optimizing, and deploying AI-driven solutions that solve real-world business problems at scale.
Key Responsibilities

Design & Develop GenAI Applications: Build scalable AI applications using Python, integrating LangChain, LangGraph, MCP, and AgentOps frameworks.
LLM Integration: Work with multiple LLM providers (Azure AI, AWS Bedrock, OpenAI, Anthropic, etc.) for text, multimodal, and agent-based workflows.
RAG Implementation: Architect and deploy Retrieval-Augmented Generation pipelines, integrating vector databases and knowledge graphs.
Fine-tuning & Model Ops: Fine-tune LLMs for domain-specific tasks, implement MLOps pipelines for continuous integration, testing, and monitoring.
Agentic AI Development: Design multi-agent systems with task orchestration, memory handling, and error recovery.
Deployment & Cloud Infrastructure: Deploy applications on AWS cloud (EC2, Lambda, S3, Bedrock, SageMaker, etc.) and Azure AI services.
Performance Optimization: Ensure model efficiency, latency reduction, and cost optimization in production environments.
Collaboration: Work closely with cross-functional teams (Data Scientists, DevOps, Product Owners) to deliver high-quality AI solutions.
Required Skills & Qualifications

Strong proficiency in Python with experience in backend development.
Hands-on experience with GenAI frameworks: LangChain, LangGraph, MCP, AgentOps.
Knowledge of RAG (Retrieval-Augmented Generation) pipelines and vector databases (Pinecone, Chroma, Weaviate, FAISS).
Experience in fine-tuning and prompt engineering for LLMs.
Strong understanding of MLOps (CI/CD for ML, model deployment, monitoring).
Experience with cloud AI platforms: Azure AI, AWS Bedrock, AWS SageMaker, GCP Vertex AI (preferred).
Knowledge of Agentic AI concepts – multi-agent orchestration, planning, memory.
Familiarity with Docker, Kubernetes, Terraform, and GitOps practices.
Strong problem-solving and debugging skills.

Benefits found in job post

Medical insurance
"""

initial_state: ResumeState = {
    "raw_resume":      resume_text,
    "raw_jd":          jd_text,
    "resume_sections": None,
    "jd_analysis":     None,
    "gap_report":      None,
    "refined_resume":  None,
    "output_pdf_path": None,
    "error":           None
}

# Execute the graph pipeline with the initial state.
result = app.invoke(initial_state)

print("\n" + "="*50)
print("PIPELINE COMPLETE")
print(f"PDF saved to : {result.get('output_pdf_path')}")
print("\nGAP REPORT:")
gp = result.get("gap_report", {})
print(f"  Matching skills : {gp.get('matching_skills', [])}")
print(f"  Missing skills  : {gp.get('missing_skills', [])}")
print(f"  Suggestions     : ")
for s in gp.get("suggestions", []):
    print(f"    — {s}")


[Resume Parser] Extracting sections...
[Resume Parser] Extracted: ['name', 'title', 'contacts', 'socials', 'summary', 'skills', 'experience', 'projects', 'education', 'certifications', 'publications']
[JD Parser] Analysing job description...
[JD Parser] Role: Sr GenAI & Agentic AI
[JD Parser] Required skills: ['Strong proficiency in Python', 'Hands-on experience with GenAI frameworks: LangChain, LangGraph, MCP, AgentOps', 'Knowledge of RAG pipelines and vector databases', 'Experience in fine-tuning and prompt engineering for LLMs', 'Strong understanding of MLOps']
[Gap Analyser] Running differential analysis...
[Gap Analyser] Matching skills : ['Strong proficiency in Python', 'Hands-on experience with GenAI frameworks: LangChain, LangGraph, MCP, AgentOps', 'Knowledge of RAG pipelines and vector databases', 'Experience in fine-tuning and prompt engineering for LLMs', 'Strong understanding of MLOps', 'Experience with cloud AI platforms: Azure AI, AWS Bedrock, AWS SageMaker, GCP Vertex AI